# Olist Seller Intelligence Report

This notebook presents an analysis of the [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data), a real-world e-commerce dataset covering orders placed between 2016 and 2018 across multiple Brazilian marketplaces.

The analysis is framed as a data consultancy project: Olist's commercial team wants to understand how sellers are performing, what's driving poor customer reviews, and where inefficiencies exist in their logistics. The goal is to produce findings and recommendations that could inform real operational decisions.

The dataset is stored in MongoDB Atlas and queried using PyMongo. Aggregation pipelines are the primary analytical tool, with results passed into Pandas dataframes for presentation and visualisation. The secondary aim of the notebook is to provide a working demonstration of MongoDB's aggregation framework, index optimisation, and Python driver usage.

# Section 1 - Setup and Data Quality Audit

The client has asked me to first verify that the data is complete and clean before any analysis. I also need to log the session and flag any known data issues for the record.

## 1.1 Connect to the database and list all collections

In [1]:
#import libraries
from pymongo import MongoClient
import pandas as pd
import os
from dotenv import load_dotenv
from pprint import pprint
import datetime
from utils import count_docs
import json

In [80]:
#load/declare constants
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
API_KEY = os.getenv("ANTHROPIC_KEY")
MISSING_CONDITIONS = [{"$in": [None, ""]}, {"$exists": False}]

In [3]:
#connect to database
client = MongoClient(MONGO_URI)
db = client["oilist-sales"]

In [4]:
#store collections as variable and view
collections = db.list_collection_names()
print(collections)

['data_audit', 'products', 'order-payments', 'geolocation', 'order-reviews', 'logistics_partners', 'category-translation', 'order-items', 'customers', 'orders', 'sellers']


All collections are present as expected.

## 1.2. Inspect one document from each collection and note the schema

In order to understand the schema properly, I will not just inspect one document as the schema can vary from document-to-document, due to MongoDB's nature as a schemaless database. I will instead take a sample of 1,000 documents from each collection, and find the union of the fields that appear across them. This gives a much better chance to capture the most commonly-appearing fields for each collection, thereby producing a more reliable outline of the schema than inspecting just one document.

In [5]:
#get sample of 100 docs
for collection in collections:
    sample = db[collection].find({}, limit = 1000)
    #find all values present in sample
    fields = {}
    for doc in sample:
        for field, value in doc.items():
            if field not in fields and value is not None:
                fields[field] = type(value).__name__
    #display schema for each collection as df
    schema = pd.DataFrame(fields.items(), columns=["field", "type"])
    print(collection)
    display(schema)

data_audit


,field,type
0,_id,ObjectId
1,auditor,str
2,date,datetime
3,details,str


products


,field,type
0,_id,ObjectId
1,product_id,str
2,product_category_name,str
3,product_photos_qty,int
4,product_weight_g,int
5,product_length_cm,int
6,product_height_cm,int
7,product_width_cm,int
8,product_description_length,int
9,product_name_length,int


order-payments


...

,field,type
0,_id,ObjectId
1,customer_id,str
2,customer_unique_id,str
3,customer_zip_code_prefix,str
4,customer_city,str
5,customer_state,str


orders


,field,type
0,_id,ObjectId
1,order_id,str
2,customer_id,str
3,order_status,str
4,order_purchase_timestamp,str
5,order_approved_at,str
6,order_delivered_carrier_date,str
7,order_delivered_customer_date,str
8,order_estimated_delivery_date,datetime


sellers


,field,type
0,_id,ObjectId
1,seller_id,str
2,seller_zip_code_prefix,str
3,seller_city,str
4,seller_state,str


When inspecting the documents, I have spotted two main errors: some spelling mistakes in field names, and data stored as incorrect types, so I will correct these.

### 1.2.1 Clean field names

In [6]:
#count docs with errors
num_name_error = db.products.count_documents({"product_name_lenght": {"$exists": True}})
num_desc_error = db.products.count_documents({"product_description_lenght": {"$exists": True}})
print(f"There are {num_name_error} products with misspelt product name, and {num_desc_error} products with misspelt product description.")

There are 32341 products with misspelt product name, and 32341 products with misspelt product description.


In [7]:
#rename misspelt fields
# db.products.update_many({}, {"$rename": {"product_name_lenght": "product_name_length", "product_description_lenght": "product_description_length"}}, upsert = True)

In [25]:
#recount to confirm changes
num_name_error_fixed = db.products.count_documents({"product_name_lenght": {"$exists": True}})
num_desc_error_fixed = db.products.count_documents({"product_description_lenght": {"$exists": True}})
print(f"There are {num_name_error_fixed} products with misspelt product name, and {num_desc_error_fixed} products with misspelt product description.")

There are 0 products with misspelt product name, and 0 products with misspelt product description.


### 1.2.2 Correct value typing errors

In [43]:
#count docs with errors
num_review_error = db["order-reviews"].count_documents({"review_answer_timestamp": {"$type": "string"}})
num_purchase_error = db.orders.count_documents({"order_purchase_timestamp": {"$type": "string"}})
num_carrier_error = db.orders.count_documents({"order_delivered_carrier_date": {"$type": "string"}})
num_customer_error = db.orders.count_documents({"order_delivered_customer_date": {"$type": "string"}})
print(f"There are {num_review_error} review answer dates stored as strings. There are also {num_purchase_error} orders, {num_carrier_error} carrier delivery dates, and {num_customer_error} customer delivery dates.")

There are 99224 review answer dates stored as strings. There are also 99441 orders, 97658 carrier delivery dates, and 96476 customer delivery dates.


In [41]:
#reassign strings to dates
db["order-reviews"].update_many({}, {"$set": {"review_answer_timestamp": {"$toDate": "review_answer_timestamp"}}})
db.orders.update_many({}, [
    {"$set": {"order_purchase_timestamp": {"$toDate": "$order_purchase_timestamp"}}}, 
    {"$set": {"order_delivered_carrier_date": {"$toDate": "$order_delivered_carrier_date"}}}, 
    {"$set": {"order_delivered_customer_date": {"$toDate": "$order_delivered_customer_date"}}}
    ])

UpdateResult({'n': 99441, 'electionId': ObjectId('7fffffff0000000000000192'), 'opTime': {'ts': Timestamp(1782763880, 880), 't': 402}, 'nModified': 99441, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1782763880, 880), 'signature': {'hash': b'\x87\x9a\xa2\xd0\xc5\xf1W\xc8\x1f;\xcc${\r\xb4\x8f\x8ff\x9d\xe6', 'keyId': 7600472395853332484}}, 'operationTime': Timestamp(1782763880, 880), 'updatedExisting': True}, acknowledged=True)

In [44]:
#recount to confirm changes
num_review_error_fixed = db["order-reviews"].count_documents({"review_answer_timestamp": {"$type": "string"}})
num_purchase_error_fixed = db.orders.count_documents({"order_purchase_timestamp": {"$type": "string"}})
num_carrier_error_fixed = db.orders.count_documents({"order_delivered_carrier_date": {"$type": "string"}})
num_customer_error_fixed = db.orders.count_documents({"order_delivered_customer_date": {"$type": "string"}})
print(f"There are {num_review_error_fixed} review answer dates stored as strings. There are also {num_purchase_error_fixed} orders, {num_carrier_error_fixed} carrier delivery dates, and {num_customer_error_fixed} customer delivery dates.")

There are 0 review answer dates stored as strings. There are also 0 orders, 0 carrier delivery dates, and 0 customer delivery dates.


## 1.3. Create a collection (with a validation schema) to record data audits

In [9]:
# # create new collection
# db.create_collection("data_audit", validator = {"$jsonSchema": {
#     "required": ["auditor", "date", "details"],
#     #validate
#     "properties": {
#         "auditor": {"bsonType": "string"},
#         "date": {"bsonType": "date"},
#         "details": {"bsonType": "string"}
#     }
# }})

In [10]:
#add audit log document to collection
# db.data_audit.insert_one(
#     {
#         "auditor": "Liam Cottrell",
#         "date": datetime.datetime.now(),
#         "details": "Created data audit collection and viewed schemas for each collection"
#     }
# )

In [11]:
#view document to confirm successful creation
db["data_audit"].find_one()

{'_id': ObjectId('6a287866482bcd769a72ecd2'),
 'auditor': 'Liam Cottrell',
 'date': datetime.datetime(2026, 6, 9, 21, 32, 38, 894000),
 'details': 'Created data audit collection and viewed schemas for each collection'}

## 1.4 Count documents in each collection and present the results

In [12]:
#declare dict to store counts
collection_count = {}
for collection in collections:
    #count docs in each collection and store
    num_docs = db[collection].count_documents({})
    collection_count[collection] = num_docs
    print(f"The {collection} collection contains {num_docs} documents.")

The data_audit collection contains 2 documents.
The products collection contains 32951 documents.
The order-payments collection contains 103886 documents.
The geolocation collection contains 1000163 documents.
The order-reviews collection contains 99224 documents.
The logistics_partners collection contains 6 documents.
The category-translation collection contains 71 documents.
The order-items collection contains 112650 documents.
The customers collection contains 99441 documents.
The orders collection contains 99441 documents.
The sellers collection contains 3095 documents.


## 1.5 Identify orders with missing customer delivery dates

In [13]:
#call function to get missing dates
missing_dates_ct, missing_dates_pct = count_docs(db, "orders", "order_delivered_customer_date", "$or", MISSING_CONDITIONS)
dates_finding = f"There are {missing_dates_ct} orders without a delivery date as confirmed by the customer, representing {round(missing_dates_pct * 100, 2)}% of all orders."
print(dates_finding)

There are 2965 orders without a delivery date as confirmed by the customer, representing 2.98% of all orders.


## 1.6 Identify reviews with no comments

In [14]:
#call function to get missing comments
missing_comments_ct, missing_comments_pct = count_docs(db, "order-reviews", "review_comment_message", "$or", MISSING_CONDITIONS)
comments_finding = f"There are {missing_comments_ct} customer reviews without a comment, representing {round(missing_comments_pct * 100, 2)}% of all reviews."
print(comments_finding)

There are 58247 customer reviews without a comment, representing 58.7% of all reviews.


## 1.7 Remove products with missing category names

In [15]:
#call function to get missing categories
missing_cats_ct, missing_cats_pct = count_docs(db, "products", "product_category_name", "$or", MISSING_CONDITIONS)
cats_finding = f"There are {missing_cats_ct} products missing a category name, representing {round(missing_cats_pct * 100, 2)}% of all products."
print(cats_finding)

There are 610 products missing a category name, representing 1.85% of all products.


In [16]:
#remove products from database
# db["products"].delete_many({"$or": [{"product_category_name": {"$in": [None, ""]}}, {"product_category_name": {"$exists": False}}]})

## 1.8 Outline order status values

Next, the client is interested in finding any orders with a status other than "delivered" or "shipped". I will count these, and list the distinct statuses present in the dataset.

In [17]:
#call function to get unfilfilled orders
unfulfilled_statuses_ct, unfulfilled_statuses_pct = count_docs(db, "orders", "order_status", "$nor", [{"$eq": "shipped"}, {"$eq": "delivered"}])
unfulfilled_finding = f"There are {unfulfilled_statuses_ct} orders that are unfulfilled (not shipped or delivered), representing {round(unfulfilled_statuses_pct * 100, 2)}% of all orders."
print(unfulfilled_finding)

There are 1856 orders that are unfulfilled (not shipped or delivered), representing 1.87% of all orders.


In [18]:
#find unique order status values
statuses = db["orders"].distinct("order_status")
print(statuses)

['approved', 'canceled', 'created', 'delivered', 'invoiced', 'processing', 'shipped', 'unavailable']


## 1.9 Create a collection for logistics partners and insert data

The client flagged that courier data had been omitted from the original dataset. They've provided a JSON file containing details of the logistics partners used for fulfilment, which will be needed later in the project. I will import this data into a new collection.

In [19]:
#import logistics partners from json file
logistics_partners = json.loads(open("logistics-partners.json").read())
#print to check
pprint(logistics_partners[0])
print(f"There are {len(logistics_partners)} logistics partners.")

{'active': True,
 'avg_delivery_days': 8,
 'carrier_code': 'COR',
 'carrier_name': 'Correios',
 'on_time_rate': 0.71,
 'regions_covered': ['SP',
                     'RJ',
                     'MG',
                     'BA',
                     'PR',
                     'RS',
                     'SC',
                     'PE',
                     'CE',
                     'GO']}
There are 6 logistics partners.


In [20]:
# #create and validate new collection
# db.create_collection("logistics_partners", validator = {"$jsonSchema": {
#     "required": ["carrier_name", "carrier_code", "regions_covered", "avg_delivery_days", "on_time_rate", "active"],
#     "properties": {
#         "carrier_name": {"bsonType": "string"},
#         "carrier_code": {"bsonType": "string"},
#         "regions_covered": {"bsonType": "array"},
#         "avg_delivery_days": {"bsonType": "int"},
#         "on_time_rate": {"bsonType": "double"},
#         "active": {"bsonType": "bool"}
#     }
# }})

In [21]:
# db.logistics_partners.insert_many(logistics_partners)

In [22]:
#view doc to check insert
pprint(db["logistics_partners"].find_one())
print(f"There are {db['logistics_partners'].count_documents({})} logistics partners.")

{'_id': ObjectId('6a38343ae62d4cd7609b6ddd'),
 'active': False,
 'avg_delivery_days': 7,
 'carrier_code': 'ROD',
 'carrier_name': 'Rodonaves',
 'on_time_rate': 0.76,
 'regions_covered': ['SP', 'MG', 'PR', 'RS', 'SC']}
There are 6 logistics partners.


## 1.10 Update audit log with findings

In [23]:
# #add new log
# db.data_audit.insert_one(
#     {
#         "auditor": "Liam Cottrell",
#         "date": datetime.datetime.now(),
#         "details": "Counted documents in each collection: \n" + str(collection_count) + "\n" + "Explored missing dates, comments, categories, and unfulfilled orders in the orders collection. Findings: \n" + dates_finding + "\n" + comments_finding + "\n" + cats_finding + "\n" + unfulfilled_finding + "\n" + "Created logistics partners collection."
#     }
# )

In [24]:
audit_check = db.data_audit.find_one(sort=[( '_id', -1)])
pprint(audit_check)

{'_id': ObjectId('6a3837f6e62d4cd7609b6de4'),
 'auditor': 'Liam Cottrell',
 'date': datetime.datetime(2026, 6, 21, 20, 13, 58, 292000),
 'details': 'Counted documents in each collection: \n'
            "{'data_audit': 1, 'products': 32951, 'order-payments': 103886, "
            "'geolocation': 1000163, 'order-reviews': 99224, "
            "'category-translation': 71, 'order-items': 112650, 'customers': "
            "99441, 'orders': 99441, 'sellers': 3095}\n"
            'Explored missing dates, comments, categories, and unfulfilled '
            'orders in the orders collection. Findings: \n'
            'There are 2965 orders without a delivery date as confirmed by the '
            'customer, representing 2.98% of all orders.\n'
            'There are 58247 customer reviews without a comment, representing '
            '58.7% of all reviews.\n'
            'There are 610 products missing a category name, representing '
            '1.85% of all products.\n'
            'There ar

# Section 2 - Customer & Order Exploration

The client wants to understand their customer base and order patterns before diving into seller performance.

## 2.1 Identify and quantify the busiest year

In [56]:
#confirm date range is correct
max_year_doc = db.orders.find().sort("order_purchase_timestamp", -1).limit(1)
max_year = max_year_doc[0]["order_purchase_timestamp"].year
min_year_doc = db.orders.find().sort("order_purchase_timestamp", 1).limit(1)
min_year = min_year_doc[0]["order_purchase_timestamp"].year
print(f"The dataset ranges from {min_year} to {max_year}.")

The dataset ranges from 2016 to 2018.


In [65]:
#count orders per year
year_counts = {}
for year in range(min_year, max_year + 1):
    year_counts[year] = db.orders.count_documents({"order_purchase_timestamp": {"$lt": datetime.datetime(year + 1, 1, 1), "$gte": datetime.datetime(year, 1, 1)}})

#find year with most orders
most_orders = max(year_counts, key = year_counts.get)
print(f"The year with the most orders is {most_orders} with {year_counts[most_orders]} orders.")

The year with the most orders is 2018 with 54011 orders.


## 2.2 Identify customers in São Paulo

In [87]:
#find count and proportion of SP customers
sp_doc_count = db["customers"].count_documents({"customer_state": "SP"})
sp_proportion = sp_doc_count / db["customers"].count_documents({})
print(f"There are {sp_doc_count} customers from the state of Sao Paulo, representing {round(sp_proportion * 100, 2)}% of all customers.")

There are 41746 customers from the state of Sao Paulo, representing 41.98% of all customers.


## 2.3 Identify late deliveries

## 2.4 Identify low-scoring reviews

In [79]:
#find reviews scoring 1 or 2
low_reviews = db["order-reviews"].find({"review_score": {"$in": [1, 2]}}, {"_id": 0,"review_score": 1, "review_comment_message": 1}).limit(10)
for doc in low_reviews:
    print(doc)
    print("------")

{'review_score': 1}
------
{'review_score': 2, 'review_comment_message': 'GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E ESSA COMPRA AGORA ME DECPCIONOU'}
------
{'review_score': 1, 'review_comment_message': 'Péssimo'}
------
{'review_score': 1, 'review_comment_message': 'Não gostei ! Comprei gato por lebre'}
------
{'review_score': 1, 'review_comment_message': 'Sempre compro pela Internet e a entrega ocorre antes do prazo combinado, que acredito ser o prazo máximo. No stark o prazo máximo já se esgotou e ainda não recebi o produto.'}
------
{'review_score': 1, 'review_comment_message': 'Nada de chegar o meu pedido.'}
------
{'review_score': 2}
------
{'review_score': 1}
------
{'review_score': 1, 'review_comment_message': 'recebi somente 1 controle Midea Split ESTILO.\r\nFaltou Controle Remoto para Ar Condicionado Consul'}
------
{'review_score': 1, 'review_comment_message': 'O produto não chegou no prazo estipulado e causou transtorno, pq programei a viagem de férias do meu filho, ba